**Exploratory Data Analysis**

In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/elections_clean.csv")
print(df.head())

In [ ]:
df.shape
# (15270, 23), meaning rows and columns

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.nunique()


In [ ]:

df[['eligible', 'days_petition_to_election']]. describe()

In [ ]:

df[['employer_has_lawyer', 'petitioner_has_lawyer', 'has_intervenor']].mean()

In [ ]:
df["census_region"].value_counts(normalize=True)

In [ ]:
df.head()

In [ ]:
# column types and missing values

df.info()

In [ ]:
# the target variable 
df['union_won'].value_counts(normalize=True)


**Figure 1:** win rate by legal representation

Question: do unions win less when the employer has a lawyer, or more when the union has one?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd 

rates = df.groupby("employer_has_lawyer")["union_won"].agg(["mean", "size"])
print(rates)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(["No lawyer", "Has lawyer"], rates["mean"])
ax.axhline(df["union_won"].mean(), linestyle="--", color="gray", label="Overall win rate")

ax.set_xlabel('Employer legal representation')        
ax.set_ylabel('Union win rate')        
ax.set_title('Union win rates decrease when employers have legal representation')         
ax.set_ylim(0, 1)
ax.legend()

plt.savefig("../figures/employer_lawyer_win_rate.png", dpi=300, bbox_inches="tight")
plt.show()

**Figure 2:**

**question2:** Test whether unions win more often when they have legal counsel (petitioner_has_lawyer).
    

In [ ]:
# group by petitioner_has_lawyer
petitioner_rates = df.groupby("petitioner_has_lawyer")["union_won"].agg(["mean", "size"])
print(petitioner_rates)


fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(["No lawyer", "Has lawyer"], petitioner_rates["mean"])
ax.axhline(df["union_won"].mean(), linestyle="--", color="gray", label="Overall win rate")
ax.set_xlabel("Petitioner (Union) legal representation")
ax.set_ylabel("Union win rate")
ax.set_title("Union win rates barely change when unions hire a lawyer") 
ax.set_ylim(0, 1)
ax.legend()
plt.savefig("../figures/union_lawyer_win_rate.png", dpi=300, bbox_inches="tight")
plt.show()

**Figure3:**

Combined the plots side by side

In [ ]:
import numpy as np

emp_rates = df.groupby("employer_has_lawyer")["union_won"].mean()
pet_rates = df.groupby("petitioner_has_lawyer")["union_won"].mean()

x = np.arange(2)
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width/2, emp_rates, width, label='Employer Lawyer')
ax.bar(x + width/2, pet_rates, width, label='Union Lawyer')

ax.axhline(df["union_won"].mean(), linestyle="--", color="gray", label="Overall win rate")
ax.set_xticks(x)
ax.set_xticklabels(["No Lawyer", "Has Lawyer"])
ax.set_xlabel("Legal representation")                    
ax.set_ylabel("Union Win Rate")
ax.set_title("Union win rates drop when employers have a lawyer, but barely change when unions do")                         
ax.set_ylim(0, 1)
ax.legend()
plt.savefig("../figures/lawyer_win_rate_comparison.png", dpi=300, bbox_inches="tight")  
plt.show()

**Figure 4:**

Part I: how big are the units? (histogram with log scale)

We chose log scale because most of units are small like median 22, however a few are huge (35,050). On a normal axis, almost all the bars would squash into one tink spike on the left. A log axis spreads them out evenly like : 1, 10, 100, 1,000, 10,000.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

bins = np.logspace(0, np.log10(df["eligible"].max()), 40)  # np.logspace(...) makes 40 bins that are evenly spaced on a log scale. normal bins would look wrong on a log axis.
ax.hist(df["eligible"].dropna(), bins=bins)
ax.set_xscale("log")

ax.set_xlabel('Eligible Voters per Unit (Log Scale)')   
ax.set_ylabel('Number of Elections')  
ax.set_title('Most Bargaining Units Are Small, but Unit Sizes Span Several Orders of Magnitude')    

plt.savefig("../figures/unit_size_hist.png", dpi=300, bbox_inches="tight")
plt.show()

Part II: do small units win more? (win rate by size group)

compute the numbers first:

In [ ]:
df["size_bin"] = pd.qcut(df["eligible"], 5)
size_rates = df.groupby("size_bin", observed=True)["union_won"].agg(["mean", "size"])
size_rates

In [ ]:
labels = ['1–8', '9–17', '18–30', '31–68', '69+']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, size_rates["mean"])
ax.axhline(df["union_won"].mean(), linestyle="--", color="gray", label="Overall win rate")

ax.set_xlabel('Bargaining Unit Size (Number of Workers)')  
ax.set_ylabel('Win Rate (Proportion of Union Victories)')  
ax.set_title('Union Win Rates Decline as Bargaining Unit Size Increases')   
ax.set_ylim(0, 1)
ax.legend()

plt.savefig("../figures/win_rate_by_unit_size.png", dpi=300, bbox_inches="tight")
plt.show()



**Figure 5**

Question: Do unions win more often when elections are held quickly?


In [ ]:
df['days_bin'] = pd.qcut(df['days_petition_to_election'], 5)
days_rates = df.groupby('days_bin', observed=True)['union_won'].agg(['mean', 'size'])
days_rates

In [ ]:
labels = ['2–22', '23–28', '29–35', '36–42', '43+']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, days_rates["mean"])
ax.axhline(df["union_won"].mean(), linestyle="--", color="gray", label="Overall win rate")

ax.set_xlabel('Days from petition to election')   
ax.set_ylabel('Union Win Rate')   
ax.set_title('Union Win Rates Decrease as the Delay Before the Election Lengthens')  
ax.set_ylim(0, 1)
ax.legend()

plt.savefig("../figures/win_rate_by_days_to_election.png", dpi=300, bbox_inches="tight")
plt.show()



Finding: When the election happens within 22 days, unions win 78.7% of the time. When it takes more than 42 days, they win only 65.1%.
Means faster elections are linked to higher union win rates. This doesn't prove that waiting causes losses, because other confounding factors—like employer anti-union campaigns, legal challenges, or employer pushback—often cause both longer delays and lower win rates (correlation does not equal causation).
Why it matters: This suggests `days_petition_to_election` could be a useful feature for predicting the outcome.

